# NPS Agent — OpenAI Direct (No Llama Stack)

Calls OpenAI's gpt-4o directly with local MCP tool execution for latency/token benchmarking against the Llama Stack variant.

**Prerequisites:**
- NPS MCP server on `localhost:3005`
- `OPENAI_API_KEY` in environment

In [1]:
import os
import json
import time
from dotenv import load_dotenv

# Load .env from parent directory (agents_tracing-eval_mlflow/.env)
env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

import mlflow
from mlflow.entities import SpanType, AssessmentSource, AssessmentSourceType
from mlflow.genai.judges import make_judge
from openai import OpenAI
from mcp.client.sse import sse_client
from mcp import ClientSession
from typing import Literal

## Configuration

In [2]:
# No Llama Stack — call OpenAI directly
# MCP URL uses localhost (no container boundary to cross)
NPS_MCP_URL = "http://localhost:3005/sse/"
MODEL_ID = "gpt-4o"
JUDGE_MODEL = "openai:/gpt-4o"

SYSTEM_PROMPT = (
    "You are a helpful National Parks Service assistant. "
    "Use the available tools to answer questions about national parks, "
    "events, activities, campgrounds, and visitor information."
)

In [3]:
db_path = os.path.join(os.getcwd(), "mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("nps-agent")
print(f"MLflow database: {db_path}")

MLflow database: /Users/sschifma/DEV/rh_repos/agents/examples/agents_tracing-eval_mlflow/nps_agent/mlflow.db


## MCP Tool Conversion

Convert MCP tool definitions to OpenAI function calling format. MCP's `inputSchema` is already JSON Schema, which maps directly to OpenAI's `parameters` field.

In [4]:
def mcp_tool_to_openai(tool) -> dict:
    """Convert an MCP Tool to OpenAI function calling format."""
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema,
        }
    }

## Agent Function

Connects to the MCP server, discovers tools, and runs a tool-calling loop against OpenAI directly. Aggregates latency and tokens across all LLM round-trips.

In [5]:
@mlflow.trace(name="query_nps_openai_direct", span_type=SpanType.AGENT)
async def query_nps(prompt: str, model: str = MODEL_ID) -> str:
    """Query the National Parks Service agent via OpenAI directly."""
    client = OpenAI()

    async with sse_client(NPS_MCP_URL) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            openai_tools = [mcp_tool_to_openai(t) for t in tools_result.tools]

            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ]
            total_input_tokens = 0
            total_output_tokens = 0
            total_tokens = 0
            iteration = 0
            start_time = time.time()

            while True:
                iteration += 1
                with mlflow.start_span(name=f"llm_call_{iteration}", span_type=SpanType.LLM) as span:
                    span.set_inputs({"model": model, "iteration": iteration, "message_count": len(messages)})

                    response = client.chat.completions.create(
                        model=model,
                        messages=messages,
                        tools=openai_tools,
                    )
                    choice = response.choices[0]

                    if response.usage:
                        total_input_tokens += response.usage.prompt_tokens
                        total_output_tokens += response.usage.completion_tokens
                        total_tokens += response.usage.total_tokens

                    span.set_outputs({
                        "finish_reason": choice.finish_reason,
                        "prompt_tokens": response.usage.prompt_tokens if response.usage else 0,
                        "completion_tokens": response.usage.completion_tokens if response.usage else 0,
                    })

                if choice.finish_reason == "stop":
                    break

                if choice.message.tool_calls:
                    messages.append(choice.message.model_dump(exclude_none=True))

                    for tool_call in choice.message.tool_calls:
                        func_name = tool_call.function.name
                        func_args = json.loads(tool_call.function.arguments)

                        with mlflow.start_span(name=f"tool_{func_name}", span_type=SpanType.TOOL) as tool_span:
                            tool_span.set_inputs({"tool": func_name, "arguments": func_args})
                            result = await session.call_tool(func_name, func_args)
                            tool_output = "".join(
                                block.text for block in result.content if hasattr(block, "text")
                            )
                            tool_span.set_outputs({"result_length": len(tool_output)})

                        messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": tool_output,
                        })

            latency_ms = (time.time() - start_time) * 1000
            mlflow.log_metrics({
                "latency_ms": latency_ms,
                "input_tokens": total_input_tokens,
                "output_tokens": total_output_tokens,
                "total_tokens": total_tokens,
            })

            return choice.message.content or ""

## Agent-as-a-Judge

In [6]:
nps_judge = make_judge(
    name="nps_agent_evaluator",
    instructions=(
        "Evaluate the NPS agent's performance in {{ trace }}.\n\n"
        "Check for:\n"
        "1. Response Quality: Did the agent correctly identify parks and provide accurate information?\n"
        "2. Tool Usage: Were the correct NPS MCP tools used (search_parks, get_park_events, etc.)?\n"
        "3. Completeness: Did the agent answer all parts of the user's question?\n\n"
        "Rate as: 'good', 'acceptable', or 'poor'"
    ),
    feedback_value_type=Literal["good", "acceptable", "poor"],
    model=JUDGE_MODEL,
)

In [7]:
def evaluate_trace(trace):
    """Run Agent-as-a-Judge evaluation and log to MLflow."""
    feedback = nps_judge(trace=trace)

    trace_id = trace.info.trace_id
    mlflow.log_feedback(
        trace_id=trace_id,
        name="nps_agent_evaluation",
        value=feedback.value,
        rationale=feedback.rationale,
        source=AssessmentSource(
            source_type=AssessmentSourceType.LLM_JUDGE,
            source_id=f"agent-as-a-judge/{JUDGE_MODEL}",
        ),
    )

    print(f"\nEvaluation: {feedback.value}")
    print(f"Rationale: {feedback.rationale}")
    return feedback

## Run Agent & Evaluate

In [8]:
prompt = "Tell me about some parks in Rhode Island, and let me know if there are any upcoming events at them."

with mlflow.start_run(run_name="nps-openai-direct-gpt-4o"):
    result = await query_nps(prompt)
    print(f"Response:\n{result}")

    mlflow.flush_trace_async_logging()

    trace_id = mlflow.get_last_active_trace_id()
    trace = mlflow.get_trace(trace_id)
    evaluate_trace(trace)

Response:
Here are some national parks in Rhode Island and their upcoming events:

### Blackstone River Valley National Historical Park
- **Old Slater Mill Tour**: Join a Park Ranger on a 30-minute guided tour of the mill that started the American Industrial Revolution. Tours occur daily at 10:30 AM, 12:30 PM, and 2:30 PM. Location: 67 Roosevelt Avenue, Pawtucket, RI.
- **First Strike Commemoration**: Commemorate the nation's first strike, with hands-on activities and weaving workshops. Location: Pawtucket, RI.
- **Take Me Fishing**: A free family fishing program every Sunday afternoon from June to August. Location: Blackstone River State Park, Lincoln, RI.
- **Nature Walk and Scavenger Hunt**: Walks for all ages with a scavenger hunt. Location: Blackstone River State Park, Lincoln, RI.

### Roger Williams National Memorial
- **Visitor Center Open**: Explore the life and legacy of Roger Williams.
- **Outdoor Yoga**: Join a free yoga class every Saturday morning.
- **Lunchtime Arts in t

10:05:12 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
10:05:12 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'



Evaluation: good
Rationale: The agent performed well in the task based on the following observations:

1. **Response Quality:** The agent correctly identified multiple parks in Rhode Island, such as the Blackstone River Valley National Historical Park, Roger Williams National Memorial, Touro Synagogue National Historic Site, and the Washington-Rochambeau Revolutionary Route National Historic Trail. It also provided accurate and detailed information about events at these parks like tours, workshops, and other community activities.

2. **Tool Usage:** The correct NPS MCP tools were used appropriately. Specifically, the agent utilized the `search_parks` tool to identify parks in Rhode Island using the state code "RI" and the `get_park_events` tool to fetch events related to these parks.

3. **Completeness:** The agent comprehensively answered all parts of the user's question by listing parks in Rhode Island and detailing upcoming events. There were no significant omissions in the provide

## View Traces in MLflow UI

Start the MLflow UI to view traces and assessments:

```bash
mlflow ui --port 5001
```

Then open http://localhost:5001 in your browser.

### Comparing Runs

1. Select the `nps-agent` experiment
2. Check both runs: `nps-openai/gpt-4o` (Llama Stack) and `nps-openai-direct-gpt-4o` (OpenAI direct)
3. Click **Compare** to see latency and token metrics side-by-side